# CAREMap · 07 Case studies: Maui, Archuleta County CO, Lake County MI

v2, step 4. Three counties, chosen to test the ranking from different angles:

- **Maui County, HI** (metro #2): is it ranked High because of the August 2023 Lahaina wildfire rather than a lasting shortage?
- **Archuleta County, CO** (rural #3, Pagosa Springs): the closest comparison to Ouray, a southwest Colorado mountain county.
- **Lake County, MI** (Baldwin): the Michigan candidate that stayed near the top under every Michigan check in notebook 04.

Each gets the same panel from the project's data (need, fit, clustering), then desk research from public reporting and local plans. Sources are listed at the end.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

s = pd.read_csv(ROOT / "data/processed/county_scores_v1.csv", dtype={"GEOID": str})
fit = pd.read_csv(ROOT / "data/processed/county_fit.csv", dtype={"GEOID": str})
clu = pd.read_csv(ROOT / "data/processed/county_clusters.csv", dtype={"GEOID": str})
mi = pd.read_csv(ROOT / "data/processed/mi_licensed_model.csv", dtype={"GEOID": str})
acs = pd.read_csv(ROOT / "data/raw/acs_2020_2024.csv", dtype={"GEOID": str})

CASES = {"15009": "Maui County, HI", "08007": "Archuleta County, CO", "26085": "Lake County, MI"}
d = (s.merge(fit[["GEOID", "fit", "fit_points", "F1_population", "F2_price_above_model", "F3_provider_pool", "F4_group_home_license"]],
             on="GEOID", how="left")
      .merge(clu[["GEOID", "lisa"]], on="GEOID", how="left")
      .merge(acs[["GEOID", "median_home_value"]], on="GEOID", how="left"))
n_group = s[s["tier"] != "Not scored"].groupby("group").size()
panel = d[d["GEOID"].isin(CASES)].set_index("GEOID").loc[list(CASES)]
panel["rank"] = panel.apply(lambda r: f"{int(r['rank_in_group'])} of {n_group[r['group']]} {r['group'].lower().replace('nonmetro', 'rural')}", axis=1)
panel["tier shown"] = np.where(panel["high_if_zero"].astype(str) == "True", "High, unconfirmed", panel["tier"])
panel["business data"] = np.where(panel["cbp_withheld_0to2"].astype(str) == "True", "withheld (0-2 employers)", "published")
show = panel[["rank", "tier shown", "childcare_score", "housing_score", "kids_under5", "kpw_low_eb", "business data",
              "rent_burden_eb", "price_to_income", "provider_rent_share", "fmr_2br", "childcare_wage", "median_home_value",
              "pop_change_pct", "fit", "fit_points", "lisa"]]
show.index = [CASES[g] for g in show.index]
show.T

,"Maui County, HI","Archuleta County, CO","Lake County, MI"
rank,2 of 1181 metro,3 of 1854 rural,56 of 1854 rural
tier shown,High,High,Medium
childcare_score,89.839119,92.286947,94.390507
housing_score,99.15326,96.871629,74.973031
kids_under5,8821,530,415
kpw_low_eb,30.624775,66.023458,82.519062
business data,published,withheld (0-2 employers),withheld (0-2 employers)
rent_burden_eb,0.558767,0.503562,0.586739
price_to_income,9.311349,5.755733,2.635567
provider_rent_share,0.862805,0.507794,0.433525


## 1. Maui County: is the rank a wildfire artifact?

The fire (August 8, 2023) destroyed more than 2,200 structures in Lahaina, including six childcare centers. The question is whether Maui's #2 metro rank reflects that shock or a shortage that was already there. The first step is to check which inputs could have been affected by the fire at all.

In [2]:
timing = pd.DataFrame([
    ["Children under 5", "ACS 2020-2024 5-year", "Mostly before (about 3.4 of 5 years)"],
    ["Childcare workers (employer)", "County Business Patterns 2023, mid-March pay period", "Before"],
    ["Childcare workers (self-employed)", "Nonemployer Statistics 2023 (tax year 2023)", "Mixed"],
    ["Rent burden, price-to-income", "ACS 2020-2024 5-year", "Mostly before"],
    ["2-bedroom rent (provider side)", "HUD Fair Market Rent FY2027", "After"],
    ["Childcare worker pay", "BLS OEWS May 2025", "After"],
], columns=["Input", "Source", "Before or after the fire"])
timing

,Input,Source,Before or after the fire
0,Children under 5,ACS 2020-2024 5-year,Mostly before (about 3.4 of 5 years)
1,Childcare workers (employer),"County Business Patterns 2023, mid-March pay period",Before
2,Childcare workers (self-employed),Nonemployer Statistics 2023 (tax year 2023),Mixed
3,"Rent burden, price-to-income",ACS 2020-2024 5-year,Mostly before
4,2-bedroom rent (provider side),HUD Fair Market Rent FY2027,After
5,Childcare worker pay,BLS OEWS May 2025,After


The childcare side is essentially pre-fire. Census employer counts come from March 2023, five months before the fire, so the fire could only have made Maui's real shortage *worse* than scored. Independently, the University of Hawai‘i's pre-fire count (July-October 2022) found 0.28 licensed seats per young child in Maui County against 0.31 statewide, and 0.07 for ages 0-2.

The one input clearly measured after the fire is the 2-bedroom Fair Market Rent. Maui's FY2027 FMR is $2,656. Its FY2023 FMR, implied by NLIHC's 2023 housing wage of $38.19, was about $1,986, a 34% rise. Honolulu's rose about 13% over the same years ($2,247 to $2,542). The test: replace Maui's rent with what it would be had it grown at Honolulu's pace, or not grown at all, and re-score the whole country.

In [3]:
scored = s[s["tier"] != "Not scored"].copy()


def score(t):
    t = t.copy()
    pct = lambda c: t.groupby("group")[c].rank(pct=True) * 100
    fam = pd.concat([pct("rent_burden_eb"), pct("price_to_income")], axis=1).mean(axis=1)   # same as notebook 02 (skips a missing value)
    t["housing_score"] = (0.5 * fam + 0.5 * pct("provider_rent_share")).groupby(t["group"]).rank(pct=True) * 100
    t["childcare_score"] = pct("kpw_low_eb")
    t["combined_score"] = np.sqrt(t["childcare_score"] * t["housing_score"])
    t["rank_in_group"] = t.groupby("group")["combined_score"].rank(ascending=False, method="min").astype(int)
    cut = t.groupby("group")["combined_score"].transform(lambda x: x.quantile(0.75))
    t["tier"] = np.select([(t["childcare_score"] >= 75) & (t["housing_score"] >= 75), t["combined_score"] >= cut],
                          ["High", "Medium"], "Low")
    return t


base = score(scored)
assert np.allclose(base["combined_score"], scored["combined_score"]), "re-scoring does not reproduce notebook 02"

FMR_FY23 = {"Maui": 38.19 * 2080 * 0.30 / 12, "Honolulu": 43.21 * 2080 * 0.30 / 12}   # from NLIHC Out of Reach 2023 housing wages
honolulu_growth = scored.loc[scored["GEOID"] == "15003", "fmr_2br"].iloc[0] / FMR_FY23["Honolulu"]
maui = scored["GEOID"] == "15009"
wage = scored.loc[maui, "childcare_wage"].iloc[0]

rows = []
for label, rent in [("As scored (FY2027 rent)", scored.loc[maui, "fmr_2br"].iloc[0]),
                    ("Rent grows at Honolulu's pace since FY2023", FMR_FY23["Maui"] * honolulu_growth),
                    ("Rent frozen at FY2023 level", FMR_FY23["Maui"])]:
    t = scored.copy()
    t.loc[maui, "provider_rent_share"] = rent * 12 / wage
    r = score(t).loc[maui].iloc[0]
    rows.append({"scenario": label, "2-bed rent": round(rent), "provider rent share": round(rent * 12 / wage, 2),
                 "housing score": round(r["housing_score"], 1), "combined": round(r["combined_score"], 1),
                 "metro rank": int(r["rank_in_group"]), "tier": r["tier"]})
print(f"Honolulu FMR growth FY2023 -> FY2027: {honolulu_growth - 1:.0%}; Maui: "
      f"{scored.loc[maui, 'fmr_2br'].iloc[0] / FMR_FY23['Maui'] - 1:.0%}")
pd.DataFrame(rows)

Honolulu FMR growth FY2023 -> FY2027: 13%; Maui: 34%


,scenario,2-bed rent,provider rent share,housing score,combined,metro rank,tier
0,As scored (FY2027 rent),2656,0.86,99.2,94.4,2,High
1,Rent grows at Honolulu's pace since FY2023,2247,0.73,97.7,93.7,3,High
2,Rent frozen at FY2023 level,1986,0.65,94.0,91.9,3,High


In [4]:
hi = base[base["USPS"] == "HI"][["county_name", "group", "rank_in_group", "tier", "childcare_score", "housing_score",
                                     "rent_burden_eb", "price_to_income", "provider_rent_share"]]
print("All Hawaii counties, as scored:")
hi.round(2)

All Hawaii counties, as scored:


,county_name,group,rank_in_group,tier,childcare_score,housing_score,rent_burden_eb,price_to_income,provider_rent_share
548,Hawaii County,Nonmetro,99,Medium,63.75,99.24,0.52,6.60,0.64
549,Honolulu County,Metro,10,High,78.24,99.49,0.58,8.45,0.83
551,Kauai County,Nonmetro,38,Medium,74.97,99.51,0.52,8.94,0.65
552,Maui County,Metro,2,High,89.84,99.15,0.56,9.31,0.86


**Verdict on Maui.** Maui's rank is not a wildfire artifact.
- **Childcare:** that side was measured before the fire, and the University of Hawai‘i's 2022 count agrees with it.
- **Housing:** even with Maui's rent set back to its pre-fire level, the housing score only falls from 99 to 94. Maui moves from #2 to #3 among metros and stays High. Hawaii's other counties score just as high on housing, so the housing pressure was already there before the fire.

The fire made both problems worse after the data was collected, so the ranking may understate Maui's current shortage. What the fire does change is fit. Maui's population is falling, and the rebuild is absorbing land, builders and recovery money, so an Ouray-style project would compete with recovery for the same resources.

## 2. Archuleta County, CO (Pagosa Springs)

**What the data says.** Rural #3 of 1,854, High and confirmed: High under both business-data assumptions, even though the Census withholds its employer childcare data. Both scores are above the 90th percentile of rural counties. The typical home is worth about $478,000, and a 2-bedroom Fair Market Rent would take about half a childcare worker's pay. Fit is *Possible* (3 of 4): growing population, prices above the model's price point, and Colorado licenses large family homes. The one miss is a thin local pool of home-based providers. On the cluster map it is a *high outlier*: High itself, with neighbors that are not.

**What local sources add.**
- **Childcare:** the shortage is long documented. A 2017 town-county work group found that the number of licensed slots would have to double to meet demand. The Early Childhood Council of Archuleta County's 2024 assessment found 6 licensed infant placements for about 100 births a year, and households spending nearly 15% of income on care.
- **Housing:** the 2025 regional Housing Needs Assessment says the county and town need at least 1,316 new units by 2035. Earlier reporting tied the squeeze to short-term rentals, which grew 87% between 2018 and 2021.

**Fit, in practice.** Archuleta already has the two partners Waterview depended on: a regional early childhood council and local governments actively funding childcare and housing. It is in Colorado, where the state Prop 123 housing fund and Rural Homes itself operate. The weak point matches the data: local coverage in 2020 described a single licensed family childcare home operating in the county. As in Ouray, recruiting and training providers would be the hardest part. The local reporting also notes limited water and sewer service outside town, which constrains where homes could go.

**Takeaway.** This is the closest match to Ouray in the High tier, both in what it needs and in what it already has.

In [5]:
print("Lake County in the Michigan checks (notebook 04):")
lake = mi[mi["GEOID"] == "26085"][["county_name", "kids_under5", "licensed_slots", "kids_per_slot_eb", "rank_proxy_mi",
                                   "rank_lic_mi", "cbp_withheld_0to2"]]
display(lake.round(2))
tr = pd.read_csv(ROOT / "data/processed/mi_tracts_licensed.csv", dtype={"GEOID": str, "county": str})
lt = tr[tr["county"] == "26085"]
print(f"Tracts: {len(lt)}; share of young children in a tract with no provider or 3+ per slot: "
      f"{lt.loc[lt['thin'], 'kids_under5'].sum() / lt['kids_under5'].sum():.0%}")

Lake County in the Michigan checks (notebook 04):


,county_name,kids_under5,licensed_slots,kids_per_slot_eb,rank_proxy_mi,rank_lic_mi,cbp_withheld_0to2
41,Lake County,415,213.0,1.94,1,3,True


Tracts: 4; share of young children in a tract with no provider or 3+ per slot: 73%


## 3. Lake County, MI (Baldwin)

**What the data says.** Nationally Medium (rural #56), but #1 in Michigan on the worker-based measure and #3 on licensed slots. It is one of the few Michigan counties near the top under both. About 73% of its young children live in census tracts with no licensed provider or 3+ children per estimated slot.

Its housing pressure is a different kind from Archuleta's or Maui's. Prices are low (a typical home is about $134,000; the price-to-income rank is middling), but renters are among the most cost-burdened in rural America (96th percentile) because incomes are low. Fit is *Weak* (2 of 4): the population is growing and Michigan licenses group homes up to 12, but the market is far below the model's price point and the local provider pool is thin.

**What local sources add.**
- **Housing:** the University of Michigan's 2025 fiscal case study describes a "housing paradox." About two-thirds of the county's homes are vacant, more than 8,000 of them seasonal, while year-round workers can't find housing. The county's 2025-2029 Strategic Housing Plan, based on a 2024 needs assessment, counts 937 units needed by 2028. Of the for-sale need, a third is entry-level under $126,000. Its goals include at least 25 new modular homes, and it lists a county land bank with properties for sale.
- **Childcare:** a 2019 state report listed Lake among ten Michigan counties with no licensed center slots for children under 30 months.

**Fit, in practice.** Two building blocks of the Rural Homes approach are already in the county plan: modular homes and land through a land bank. But new homes at Rural Homes' Colorado prices ($275k-$480k) would cost two to three times the local market, and the local need is mostly for entry-level homes. A Lake County version would look different:
- a modular or rehabbed home at local prices
- held by a nonprofit or land bank as a rental, as Mission Driven Finance does in Ouray
- licensed as a group home for up to 12 children

**Takeaway.** Lake shows that the overlap of the two shortages exists in low-cost rural Michigan too. The Ouray model would need to be adapted rather than copied: cheaper construction and a rental structure instead of new homes for sale.

## What the three cases say about CAREMap

1. **The ranking holds up where it was most likely to be wrong.** Maui's #2 rank was the obvious candidate for a one-time shock, and it isn't one.
2. **Need and fit really are different questions.** Archuleta and Maui are both High, but Archuleta looks like a candidate for the Ouray model and Maui doesn't right now.
3. **The model travels, but not unchanged.** Lake County has the overlap but not the price structure. In a low-cost market, the useful version of the idea is "licensed childcare homes held as long-term rentals" rather than "new homes for sale."

## Sources

- University of Hawai‘i Center on the Family, *Access to Early Childhood Care and Education: Maui County* (2024; data collected July-October 2022).
- Honolulu Civil Beat, "Lahaina families frustrated over lack of child care options" (Nov 2023); Maui Now, West Maui Early Childhood Collective (Sep 2024).
- UHERO Maui Recovery Survey and dashboard (2024-2026).
- National Low Income Housing Coalition, *Out of Reach* 2023 (Hawaii housing wages); Honolulu Star-Advertiser (Oct 2026) for Maui's FY2027 FMR.
- Pagosa Springs SUN: Early Childhood Council of Archuleta County assessment (Apr 4, 2024); Wings Early Childhood Center on the 2017 work group.
- Pagosa Daily Post, "Wings Over Pagosa, Part Five" (2020) on licensed family childcare homes.
- Archuleta County & Town of Pagosa Springs, *Regional Housing Needs Assessment* (Apr 2025); Colorado Sun (Oct 2021) on short-term rentals.
- University of Michigan CLOSUP, *Rural Economic Diversification and Fiscal Stability: A Case Study of Lake County, MI* (Aug 2025).
- Lake County Housing Coalition, *Strategic Housing Plan 2025-2029*.
- Michigan Department of Education press release on *Changes in Michigan's Child Care Landscape* (Feb 2019).